In [18]:
## requires 
# pip install snowflake-connector-python[pandas]
# pip install sqlalchemy

# save the columns and tables information schemas from sqlserver to csvs
# save the ddl scripts from sqserver to txt file

Run Cells Sequentially
Each cell is numbered in the order they should be run in

In [19]:
#import statements
import snowflake.connector
import pandas as pd
from sqlalchemy import VARCHAR, false, table
import os
import json
import re
from configparser import ConfigParser

In [20]:
#set custom variables
kcard = '23527'
user_email = 'minh.cao@us.tel.com'

project = 'csdpro'.lower()
project_c = project.capitalize()
project_u = project.upper()

server = 'TEHAUS0164\\SQL2017'
db = 'CSDPro2'


#remaining variables are auto set
lzSchema = project_u
odsSchema = project_u
pubSchema = "ODS_" + project_u

LZdb = "LZ_DEV"
ODSdb = "LZ_DEV"
overwriteDDLs = True

repo_path = "C:\\Sources\\Repos\\EnterpriseCloud\\"
files_path = repo_path + "Notebooks\\"+project_c+"ViewTemplates\\"

#snow file paths
snowLZ_path = repo_path+"Snow\\K."+kcard+".1__"+project+"base_LZTables.sql" 
snowODS_path = repo_path+"Snow\\K."+kcard+".2__"+project+"base_ODSTables.sql"

# sqlserver info, may need to be renamed
tablespath = files_path + "tables.csv"
colspath = files_path + "columns.csv"
ddl_path = files_path + "ddls.txt"

teltools_path = repo_path+'Configuration\\TELTools.config'
tt_test_path = repo_path+'PowerShell\\TELTools-TestRunner.ps1'

fileuploader_path = repo_path+"Configuration\\FileUploader_"+project_c+".config"

typeDict = {'nvarchar':'VARCHAR(16777216)',
            'numeric':'NUMBER(38,0)',
            'datetime':'TIMESTAMP_LTZ(9)',
            'datetime2':'TIMESTAMP_LTZ(9)',
            'char':'VARCHAR(16777216)',
            'varchar':'VARCHAR(16777216)',
            'boolean':'BOOLEAN',
            'decimal':'NUMBER(38,10)',
            'float':'NUMBER(38,10)',
            'str':'VARCHAR(16777216)',
            'bit':'BOOLEAN',
            'uniqueidentifier':'VARCHAR(16777216)',
            'int':'NUMBER(38,0)',
            'money':'NUMBER(38,10)',
            'date':'TIMESTAMP_LTZ(9)',
            'nchar':'VARCHAR(16777216)',
            'smallint':'NUMBER(38,0)',
            'bigint':'NUMBER(38,0)',
            'text':'VARCHAR(16777216)',
            }

In [21]:
#utilities
def getFile(path, split=True):
    f = open(path,'r')
    if split:
      filein = f.readlines()
    else:
      filein = f.read()
    f.close()
    return filein

def writeFile(path,fileOut):
    f = open(path,'w')
    f.write(fileOut)
    f.close()

def get_tt_config(server, db, lzSchema, table, columns):
  columns_str = '], ['.join(columns)
  config = """,
      \""""+lzSchema.upper()+'-'+table+"""\": {
        "CacheDataSet": "No",
        "AuthenticationType": "SQLServer",
       "Source": "CSDPro",
        "Securables": {
          "UserName": "TELUSDW",
          "PasswordName": "CSDPro-Password"
        },
        "Parameters": [
          [
            {
              "Query": "SELECT ["""+columns_str+"""] FROM [dbo].["""+table+"""]"
            },
            {
              "ServerInstance": "TEHAUS0164\\SQL2017"
            },
            {
              "Database": \""""+db+"""\"
            }
          ]
        ],
        "OutputFile": {
          "FileDirectoryExtension": \""""+lzSchema+"""\\\\",
          "FileName": \""""+table+"""\",
          "Format": "json",
          "JSonDepth": 1
        }
      }"""
  
  return config


In [22]:
# adding tables to teltools file

# read csv template
df_tables = pd.read_csv(tablespath)
tables = df_tables['TABLE_NAME']
df_columns = pd.read_csv(colspath)

new_configs = ''
test_runs = ''

current_tt = getFile(teltools_path, split=False)

for table in tables:
    if project_u+"-"+table not in current_tt:
        table_info = df_columns[df_columns.TABLE_NAME == table]
        columns = list(table_info.COLUMN_NAME)
        new_configs += get_tt_config(server, db, lzSchema, table, columns)
        test_runs += "\nWrite-TELDataSet   -SourceName 'SQLStatements'      -DataSetName '"+lzSchema+'-'+table+"'   		-DataSetData $RetrievedData;"

txt = ''.join(getFile(teltools_path))
start_idx = txt.find('\n  "SQLStatements": {')
end_idx = start_idx + txt[start_idx:].find('\n  }')-6
new_txt = txt[:end_idx]+new_configs+txt[end_idx:]
writeFile(teltools_path, new_txt)

# adding tables to teltools-testrunner script file
with open(tt_test_path, 'a+') as file:
    file.write(test_runs)

# to test in dev run the teltools-testrunner.ps1 ensuring you have the path
#  C:\Temp\SnowflakeRootDirectory\<<project name>> created

In [23]:
# create snow file for LZ tables from tables csv

df_tables = pd.read_csv(tablespath)
tables = df_tables['TABLE_NAME']
df_columns = pd.read_csv(colspath)

if overwriteDDLs:
    ddlAction = """create or replace TABLE """ + lzSchema + "." + tables
else:
    ddlAction = """create TABLE """ + lzSchema + "." + tables + """ if not exists"""
    
#generate DDL
df_tables['lz_ddl'] = ddlAction + """ (\
_FILENAME VARCHAR(16777216),\
_FILEDATE TIMESTAMP_NTZ(9),\
_UPLOADDATE TIMESTAMP_NTZ(9),\
JSONFIELD VARIANT\
)COMMENT='Model """ + lzSchema + """ """ + tables + """ file.';"""

#write out LZ database table ddl commands to snow file
outstr = ""
outstr = "USE DATABASE <<LZ>>;\n"
for idx, row in df_tables.iterrows():
    outstr += row['lz_ddl'] + """
"""
print(outstr)
writeFile(snowLZ_path,outstr)

USE DATABASE <<LZ>>;
create or replace TABLE CSDPRO.INCIDENTS (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model CSDPRO INCIDENTS file.';
create or replace TABLE CSDPRO.BAAN_BP (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model CSDPRO BAAN_BP file.';
create or replace TABLE CSDPRO.BAAN_FAB (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model CSDPRO BAAN_FAB file.';
create or replace TABLE CSDPRO.BU (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model CSDPRO BU file.';
create or replace TABLE CSDPRO.INCIDENT_CLASSIFICATION (_FILENAME VARCHAR(16777216),_FILEDATE TIMESTAMP_NTZ(9),_UPLOADDATE TIMESTAMP_NTZ(9),JSONFIELD VARIANT)COMMENT='Model CSDPRO INCIDENT_CLASSIFICATION file.';
create or replace TABLE CSDPRO.INCIDENT_STATUS (_FI

In [24]:
# create the fileuploader.config entries 
# (remove existing entries if you need to overwrite them in teh fileuploader.config file)

#read existing file
from this import d


configFileList = getFile(fileuploader_path)

# create new list, remove empty lines
newFileList = []
for l in configFileList: 
    if l != "":
        newFileList.append(l)
  
# remove last line
newFileList = newFileList[:-1]

# create a working copy of df_tables to build config
df_tables_config = df_tables.copy(deep=True)

#check to see if entry exists in file uploader config, if it does drop it from the df
for idx, row in df_tables_config.iterrows():
    dfTableName = row['TABLE_NAME']  
    config_exists = False
    for tn in newFileList:
        if dfTableName in tn:
            config_exists = True
    if config_exists:
        print('Table already in file uploader config, dropping',dfTableName,idx)        
        df_tables_config.drop(idx, inplace=True)
            
#create the new config entries
confstr = ""

for l in newFileList:
    confstr += l

confstr = confstr.rstrip()
confstr = confstr.rstrip('\r\n')
confstr = confstr.rstrip()
for idx, row in df_tables_config.iterrows():
    confTableName = row['TABLE_NAME'] 
    confFileName = row['TABLE_NAME']
    confName = row['TABLE_NAME']
    confstr += """,
    \"""" + confName + """\": {
        "SNOWFLAKE_SCHEMA_NAME": \"""" + lzSchema + """\",
        "SNOWFLAKE_DATABASE_NAME": "<<LZ>>",
        "SNOWFLAKE_WAREHOUSE_NAME": "Import_File_WH",
        "SNOWFLAKE_ROLE_NAME": "<<SYSADMIN>>",
        "SNOWFLAKE_TABLE_NAME": \"""" + confTableName + """\",
        "SNOWFLAKE_FILE_FORMAT_NAME": "CONTROL.POWERSHELL_JSON",
        "SNOWFLAKE_STAGE_TYPE": "NAMED",
        "SNOWFLAKE_STAGE_NAME": \"""" + lzSchema + """_STG",
        "FILE": "F:\\\\Shares\\\\Files\\\\"""+lzSchema+"""\\\\""" + confFileName + """.*.json",
        "SPLIT_FILE": false
    }"""
    
# add final bracket
confstr += """
}"""

# write out new config file
writeFile(fileuploader_path,confstr)

Table already in file uploader config, dropping INCIDENTS 0
Table already in file uploader config, dropping BAAN_BP 1
Table already in file uploader config, dropping BAAN_FAB 2
Table already in file uploader config, dropping BU 3
Table already in file uploader config, dropping INCIDENT_CLASSIFICATION 4
Table already in file uploader config, dropping INCIDENT_STATUS 5
Table already in file uploader config, dropping LOCATION_DESCRIPTION 6
Table already in file uploader config, dropping LOCATION_TYPE 7
Table already in file uploader config, dropping PERSON 8
Table already in file uploader config, dropping PERSON_CLASSIFICATION 9
Table already in file uploader config, dropping PERSON_FUNCTION 10
Table already in file uploader config, dropping SMBWA_A 11
Table already in file uploader config, dropping SMBWA_Q 12
Table already in file uploader config, dropping STATUS_HISTORY 13
Table already in file uploader config, dropping TEL_OFFICE 14
Table already in file uploader config, dropping TEL_S

In [25]:
#**************************** LZ to ODS dbt model Utility**********************
#helper functions for DBT model

def getJinjaConfig(strategy,view,unique_keys):

	if unique_keys:
		jinjaConfig = """
{{ config(
	materialized='incremental',
	incremental_strategy='"""+ strategy +"""',
	alias='""" + view.upper() + """',
	unique_key = ['"""+ unique_keys +"""'],
	transient=false
) }} """

	else:
		jinjaConfig = """
{{ config(
	materialized='incremental',
	incremental_strategy='"""+ strategy +"""',
	alias='""" + view.upper() + """',
	transient=false
) }} """

	return jinjaConfig

#CREATE DBT MODEL
#(inputs are column list for source_cte,transform_cte and tablename )
def getDbtCode(col_s_string,col_t_string,col_type_data_str,tablename,schema,jconfig,ifMerge):

	DBT_CODE = """

/************
Model file generated from """+project+"""DbtTemplateUtility.ipynb 
*************/

"""+ jconfig +"""

with source_data as (
	SELECT
		""" +col_s_string+ """
	FROM """ + "{{ source('"+ schema + "','" + tablename.upper() + "') }}" """
	"""+ ifMerge +"""
),

transform_data as (
	SELECT 
		""" +col_t_string+ """,
		'{{ invocation_id }}' AS AUDIT_LOG_ID,
		SHA2('hash_me') AS _ROW_HASH -- Adds column to table on full refresh.
	FROM source_data
),

type_data as (
	SELECT
		"""+col_type_data_str+""",
		CAST(AUDIT_LOG_ID AS VARCHAR(128)) AS AUDIT_LOG_ID,
		CAST(_ROW_HASH AS VARCHAR(128)) AS _ROW_HASH
	FROM transform_data
)

SELECT * FROM type_data
"""

	return DBT_CODE


#DBT CODE FOR INSERT-OVERWRITES
def getDbtCode_insertoverwrite(col_s_string,col_t_string,col_type_data_str,tablename,schema,jconfig):

	DBT_CODE_I = """

/************
Model file generated from """+project+"""DbtTemplateUtility.ipynb 
*************/

"""+ jconfig +"""

with source_data as (
	SELECT
		""" +col_s_string+ """
	FROM """ + "{{ source('"+ schema + "','" + tablename.upper() + "') }}"  """
    WHERE _FILEDATE = (SELECT MAX(_FILEDATE) FROM """ + "{{ source('"+ schema + "','" + tablename + "') }}"  """)
),

transform_data as (
	SELECT 
		""" +col_t_string+ """,
		'{{ invocation_id }}' AS AUDIT_LOG_ID
	FROM source_data
),

type_data as (
	SELECT
		"""+col_type_data_str+""",
		CAST(AUDIT_LOG_ID AS varchar(128)) AS AUDIT_LOG_ID
	FROM transform_data
)

SELECT * FROM type_data
"""
	
	return DBT_CODE_I


#CREATE SCHEMA FILE
def createSchema(df_tables_test,parentDB,lzSchema,odsSchema,schemaPath):
        schema_str = """version: 2\nsources:\n  - name: """ + odsSchema + """\n    description: Landing Zone - """ + odsSchema + """ Schema\n    database: "{{ env_var('SNOWFLAKE_"""+parentDB+"""') }}"\n    schema: """+ lzSchema + """\n    tables:"""

        df_tables_test['TABLE_NAME'] = df_tables_test['TABLE_NAME'].str.upper()

        df_tables_test['schema'] = """\n      - name: """ + df_tables_test['TABLE_NAME'] + """\n        identifier: """ + df_tables_test['TABLE_NAME'] + """\n        description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        for idx, r in df_tables_test.iterrows():
            schema_str += r['schema'] + """
        """
        df_tables_test['models'] = """\n  - name: """ + df_tables_test['TABLE_NAME'] + """\n    identifier: """ + df_tables_test['TABLE_NAME'] + """\n    description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str_t ="""\nmodels:"""
        for idx, r in df_tables_test.iterrows():
            schema_str_t += r['models'] + """
        """
        final_schema = schema_str + schema_str_t
        	
        path_create = schemaPath
        filename = "schema.yml"
        path_write = path_create +"\\"+ filename

        writeFile(path_write,final_schema)
		

In [26]:
#append lz_to_ods schema.yml file
dbt_path = repo_path + "dbt\\lz_to_ods\\models\\"+project+"\\"
tables = pd.DataFrame(df_tables["TABLE_NAME"])


schema_path = dbt_path + "schema.yml"
existing_schemas = []
#if schema file exists
#add to list
if os.path.exists(schema_path):
    schema_file = getFile(schema_path, split=False)
    for model in re.findall('\n      - name: (.*?)\n', schema_file):
        if model not in list(tables['TABLE_NAME'].str.upper()):
            existing_schemas.append(model)

#remove any duplicates
existing_schemas = list(dict.fromkeys(existing_schemas))
#convert to dataframe
es = pd.DataFrame({'TABLE_NAME':existing_schemas})
#merge lists
tables = pd.concat([tables,es],ignore_index=True)
createSchema(tables,'LZ',lzSchema,lzSchema,dbt_path)

In [27]:
# extracts tables primary keys from scripts.txt file

ddls = getFile(ddl_path, split=False)

groups = re.findall('CREATE TABLE \[dbo\]\.\[(?:.|\n)*?GO', ddls)

pks = {}
working = []
for group in groups:
    table_name = re.search('CREATE TABLE \[dbo\]\.\[(.*)\]\(', group).groups()[0]
    pk_match = re.search('CONSTRAINT \[.*\] PRIMARY KEY *(?:CLUSTERED *)?\n*\(((?:.|\n)*?)\)', group)
    if pk_match:
        pks[table_name] = re.findall('\[(.*)\]', pk_match.groups()[0])

pks

{'ACTION_ITEM': ['ACTION_ITEM_ID'],
 'ACTION_ITEM_ATTACHMENT': ['ACTION_ITEM_ATTACHMENT_ID'],
 'ACTION_ITEM_COMMENT': ['ACTION_ITEM_COMMENT_ID'],
 'ATTACHMENT': ['ATTACHMENT_ID'],
 'BAAN_BP': ['BAAN_BP_ID'],
 'BAAN_FAB': ['BAAN_FAB_ID'],
 'BODY_PART': ['BODY_PART_ID'],
 'BU': ['BU_ID'],
 'CONTRIBUTING_FACTOR': ['CONTRIBUTING_FACTOR_ID'],
 'CONTRIBUTING_FACTOR_TYPE': ['CONTRIBUTING_FACTOR_TYPE_ID'],
 'CORRECTIVE_ACTIONS': ['CORRECTIVE_ACTION_ID'],
 'COUNTRY': ['COUNTRY_ID'],
 'DISTRIB_RECIP_REPORT_EMAIL': ['DISTRIB_RECIP_REPORT_EMAIL_ID'],
 'DISTRIBRECIPIENT_BAANBP': ['DISTRIBRECIPIENT_BAANBP_ID'],
 'DISTRIBRECIPIENT_BAANFAB': ['DISTRIBRECIPIENT_BAANFAB_ID'],
 'DISTRIBRECIPIENT_BU': ['DISTRIBRECIPIENT_BU_ID'],
 'DISTRIBRECIPIENT_OFFICE': ['DISTRIBRECIPIENT_OFFICE_ID'],
 'DISTRIBRECIPIENT_TELSUBSIDIARY': ['DISTRIBRECIPIENT_TELSUBSID_ID'],
 'DISTRIBUTION_LIST': ['DISTRIBUTION_LIST_ID'],
 'DISTRIBUTION_RECIPIENT': ['DISTRIBUTION_RECIPIENT_ID'],
 'EHS_INCIDENT_CLASSIFICATION': ['EHS_INCIDEN

In [28]:
# create snow file for ODS tables from columns csv

df_tables = pd.read_csv(tablespath)
tables = df_tables['TABLE_NAME']
df_columns = pd.read_csv(colspath)

chars_path = repo_path + "Notebooks\\Sf_characters.csv"
sf_characters = pd.read_csv(chars_path)

# read csv template
df_cols = pd.read_csv(colspath)

outstr = "USE DATABASE <<ODS>>;\n"

for table in tables:
    
    csvData = df_cols[df_cols['TABLE_NAME']==table].copy(deep=True).reset_index(drop=True)
    
    #loop through characters df and replace any illegal characters
    for idx,row in sf_characters.iterrows():
        csvData['COLUMN_NAME']  = csvData['COLUMN_NAME'].str.replace(row['OLD'],row['NEW'])
    
    csvData = csvData.replace({'DATA_TYPE':typeDict})
    columnList = ''.join([row["COLUMN_NAME"] + ' ' + row["DATA_TYPE"] + ",\n" for idx,row in csvData.iterrows()])

    if overwriteDDLs:
        ddlAction = 'create or replace TABLE ' + odsSchema + '.' + table + ' (\n'
    else:
        ddlAction = 'create TABLE ' + odsSchema  + ' if not exists (\n'
    
    if table in pks:
        controlColumns = '_FILEDATE TIMESTAMP_NTZ(9),\n_FILENAME varchar(16777216),\nAUDIT_LOG_ID varchar(128),\n_ROW_HASH varchar(128)\n);\n\n'
    else:
        controlColumns = 'AUDIT_LOG_ID varchar(128));\n\n'

    outstr +=  ddlAction + columnList + controlColumns

print(outstr)
writeFile(snowODS_path,outstr)

USE DATABASE <<ODS>>;
create or replace TABLE CSDPRO.INCIDENTS (
INCIDENT_ID NUMBER(38,0),
PERSON_CLASSIFICATION_ID NUMBER(38,0),
MACHINE_ID NUMBER(38,0),
PERSON_FUNCTION_ID NUMBER(38,0),
BU_ID NUMBER(38,0),
REPORT_TYPE_ID NUMBER(38,0),
TASK_ID NUMBER(38,0),
TREATMENT_CENTER_TYPE_ID NUMBER(38,0),
LOCATION_TYPE_ID NUMBER(38,0),
OTHER_LOCATION_TYPE VARCHAR(16777216),
LOCATION_DESCRIPTION_ID NUMBER(38,0),
INCIDENT_TEL_SUBSIDIARY_ID NUMBER(38,0),
VICTIM_TEL_SUBSIDIARY_ID NUMBER(38,0),
TEL_OFFICE_ID NUMBER(38,0),
BAAN_BP_ID VARCHAR(16777216),
BAAN_FAB_ID VARCHAR(16777216),
INCIDENT_CLASSIFICATION_ID NUMBER(38,0),
PERSON_ID NUMBER(38,0),
INVESTIGATION_TYPE_ID NUMBER(38,0),
INCIDENT_DATE TIMESTAMP_LTZ(9),
INCIDENT_TIME VARCHAR(16777216),
INCIDENT_TITLE VARCHAR(16777216),
INCIDENT_DETAILS VARCHAR(16777216),
BUSINESS_TRIP NUMBER(38,0),
PERSONAL_INJURY NUMBER(38,0),
DAMAGE NUMBER(38,0),
RESULTING_INJURY NUMBER(38,0),
RISK_OF_INJURY NUMBER(38,0),
RULE_VIOLATION NUMBER(38,0),
TEL_EQUIPMENT_INVOLVE

In [17]:
# Build lz_to_ods dbt models from view templates

tables = df_tables['TABLE_NAME']

# read csv template file and build list of required inputs
for table in tables:

    csvData = df_cols[df_cols['TABLE_NAME']==table].copy(deep=True).reset_index(drop=True).replace({'DATA_TYPE':typeDict})
    
    csvData["columnList"] = csvData["TABLE_NAME"] + " " + csvData["DATA_TYPE"] + ","
    
    #empty list for transform_cte,source_cte,type_data_cte columns 
    col_t = []
    col_s = []
    col_source = []
    col_type_data = []
    
    #type data columns
    for idx,row in csvData.iterrows():
        col_type_data.append('CAST("' + row["COLUMN_NAME"] + '" AS ' + row["DATA_TYPE"] + ') AS "' + row["COLUMN_NAME"].upper() + '"')
        col_t.append(row["COLUMN_NAME"])
        if row["DATA_TYPE"] == 'TIMESTAMP_LTZ(9)':
            col_s.append("REGEXP_REPLACE(JSONFIELD:\""+ row["COLUMN_NAME"] +"\", '\/Date\\\(|\\\)\/', '')::TIMESTAMP AS "+ row["COLUMN_NAME"])  
        else:
            col_s.append('JSONFIELD:"' + row["COLUMN_NAME"] + '"::STRING AS "' + row["COLUMN_NAME"].upper() + '"')  

    col_type_data_str = ', \n \t \t'.join(col_type_data)
    col_type_data_str = col_type_data_str.upper()

    # column string for transform cte
    col_t_string = '"' + '", \n \t \t"'.join(col_t) + '"'
    col_t_string = col_t_string.upper()
                 
    col_s_string = ', \n \t \t'.join(col_s)
    dbt_strategy = 'incremental'
    
    if table in pks:
        unique_fields = pks[table]    
        unique_fields_str = "','".join(unique_fields)    
        
        dbt_strategy = 'merge'
        
        unique_fields_qualify = ['JSONFIELD:"{0}"::STRING'.format(k) for k in unique_fields]
        unique_fields_qualify_str = ",".join(unique_fields_qualify)
        
        ifMerge = """{% if is_incremental() %}WHERE  _FILENAME > IFNULL((SELECT MAX(_FILENAME) FROM {{ this }}),'1970-01-01'){% endif %}
    QUALIFY (ROW_NUMBER() OVER (PARTITION BY """+unique_fields_qualify_str+"""  ORDER BY _FILENAME DESC,_FILEDATE DESC )) = 1"""
        
        jinjaConfig = getJinjaConfig(dbt_strategy,table.upper(),unique_fields_str)
        dbt_model = getDbtCode(col_s_string,col_t_string,col_type_data_str,table.upper(),lzSchema,jinjaConfig,ifMerge)

    else:
        dbt_strategy = 'insert_overwrite'
        unique_fields_str = ''
        
        jinjaConfig = getJinjaConfig(dbt_strategy,table.upper(),unique_fields_str)
        dbt_model = getDbtCode_insertoverwrite(col_s_string,col_t_string,col_type_data_str,table.upper(),lzSchema,jinjaConfig)

    #create dbt file
    dbt_path = repo_path + "dbt\\lz_to_ods\\models\\"+project+"\\"
    filename = table + ".sql"
    path_write = dbt_path + filename

    writeFile(path_write,dbt_model)
    print('created file',filename)


created file INCIDENTS.sql
created file BAAN_BP.sql
created file BAAN_FAB.sql
created file BU.sql
created file INCIDENT_CLASSIFICATION.sql
created file INCIDENT_STATUS.sql
created file LOCATION_DESCRIPTION.sql
created file LOCATION_TYPE.sql
created file PERSON.sql
created file PERSON_CLASSIFICATION.sql
created file PERSON_FUNCTION.sql
created file SMBWA_A.sql
created file SMBWA_Q.sql
created file STATUS_HISTORY.sql
created file TEL_OFFICE.sql
created file TEL_SUBSIDIARY.sql


In [29]:
#append ods_to_pub schema.yml file
dbt_path = repo_path + "dbt\\ods_to_pub\\models\\"+project+"\\"
tables = pd.DataFrame(df_tables["TABLE_NAME"])


schema_path = dbt_path + "schema.yml"
existing_schemas = []
#if schema file exists
#add to list
if os.path.exists(schema_path):
    schema_file = getFile(schema_path, split=False)
    for model in re.findall('\n      - name: (.*?)\n', schema_file):
        if model not in list(tables['TABLE_NAME'].str.upper()):
            existing_schemas.append(model)

#remove any duplicates
existing_schemas = list(dict.fromkeys(existing_schemas))
#convert to dataframe
es = pd.DataFrame({'TABLE_NAME':existing_schemas})
#merge lists
tables = pd.concat([tables,es],ignore_index=True)
createSchema(tables,'ODS',odsSchema,pubSchema,dbt_path)

In [30]:
## create pub ODS_ views

%run ..\\Python\\dbtTools\\dbtGeneratePubViews.py "ODS_DEV" "$project_u" "C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models" "$user_email"


Arguments passed: DB: ODS_DEV  Schema: CSDPRO Path: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models user: minh.cao@us.tel.com
Schema.yml already exists, delete to refresh.
ODS_DEV ODS {{ source('ODS_CSDPRO','BAAN_FAB') }}
v_ model created: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models\csdpro\v_BAAN_FAB.sql
ODS_DEV ODS {{ source('ODS_CSDPRO','PERSON') }}
v_ model created: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models\csdpro\v_PERSON.sql
ODS_DEV ODS {{ source('ODS_CSDPRO','PERSON_CLASSIFICATION') }}
v_ model created: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models\csdpro\v_PERSON_CLASSIFICATION.sql
ODS_DEV ODS {{ source('ODS_CSDPRO','PERSON_FUNCTION') }}
v_ model created: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models\csdpro\v_PERSON_FUNCTION.sql
ODS_DEV ODS {{ source('ODS_CSDPRO','SMBWA_A') }}
v_ model created: C:\Sources\Repos\EnterpriseCloud\dbt\ods_to_pub\models\csdpro\v_SMBWA_A.sql
ODS_DEV ODS {{ source('ODS_CSDPRO','SMBWA_Q') }}
v_ model crea